# KMeans 总结笔记

## 1. 一句话定义

K-means 是一种基于距离的无监督聚类算法：给定簇数量 `K`，反复分配样本并更新质心，使簇内平方距离之和尽量小。

```text
输入：只有特征矩阵 X
输出：每个样本的簇编号 + K 个质心
```

In [1]:
import os

# sklearn 1.7.x 在 Windows + MKL 的小数据集上建议限制 OpenMP 线程数
os.environ["OMP_NUM_THREADS"] = "1"

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans

np.set_printoptions(precision=3, suppress=True)

from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 2. 核心公式

分配样本：

$$
c_i=\arg\min_k\lVert x_i-\mu_k\rVert_2^2
$$

更新质心：

$$
\mu_k=\frac{1}{|C_k|}\sum_{x_i\in C_k}x_i
$$

目标函数：

$$
J=\sum_{k=1}^{K}\sum_{x_i\in C_k}\lVert x_i-\mu_k\rVert_2^2
$$

`J` 也叫 SSE、WCSS，在 sklearn 中对应 `inertia_`。

## 3. 完整工作流

```text
1. 明确聚类目标
2. 选择有意义的数值特征
3. 清理缺失值和异常值
4. 检查特征尺度并标准化
5. 用肘部法、轮廓系数和业务知识选择 K
6. 使用 k-means++、多次 n_init 训练
7. 检查 inertia、轮廓系数、簇大小和质心
8. 更换随机种子或样本检查稳定性
9. 给簇添加业务解释
10. 保存 scaler 与模型，统一处理新样本
```

## 4. sklearn 最小模板

In [2]:
X_demo, _ = make_blobs(n_samples=300, centers=3, random_state=22)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_demo)

model = KMeans(
    n_clusters=3,
    init="k-means++",
    n_init=10,
    random_state=22,
)
labels = model.fit_predict(X_scaled)

centers_original = scaler.inverse_transform(model.cluster_centers_)

print("inertia_:", round(model.inertia_, 3))
print("silhouette:", round(silhouette_score(X_scaled, labels), 4))
print("簇大小:", np.bincount(labels))
print("原始单位质心:")
print(centers_original)

inertia_: 66.925
silhouette: 0.5091
簇大小: [ 97 100 103]
原始单位质心:
[[-6.794 -3.172]
 [-1.565  7.315]
 [-5.544 -0.389]]


## 5. 主要参数速查

| 参数 | 作用 | 学习阶段建议 |
|---|---|---|
| `n_clusters` | 簇数量 K | 用指标和业务共同决定 |
| `init` | 初始化方法 | 优先 `k-means++` |
| `n_init` | 初始化尝试次数 | 显式设置 `10` |
| `max_iter` | 单次最多迭代轮数 | 默认通常足够 |
| `tol` | 收敛容差 | 默认通常足够 |
| `random_state` | 固定随机过程 | 学习和复现实验时设置 |

主要属性：

| 属性 | 含义 |
|---|---|
| `labels_` | 训练样本簇编号 |
| `cluster_centers_` | 最终质心 |
| `inertia_` | 簇内平方和 |
| `n_iter_` | 迭代次数 |

## 6. 如何选择 K

| 方法 | 看什么 | 局限 |
|---|---|---|
| 肘部法 | inertia 下降开始变缓的位置 | 肘部可能不明显 |
| 轮廓系数 | 簇内紧密与簇间分离 | 偏好几何上分离清晰的簇 |
| 稳定性 | 换样本或随机种子后是否相似 | 需要重复实验 |
| 业务知识 | 是否可解释、可行动 | 依赖领域背景 |

不能只看 `inertia_`，因为它会随 K 增大而下降。

## 7. KNN 与 K-means 对比

| 对比项 | KNN | K-means |
|---|---|---|
| 学习类型 | 有监督 | 无监督 |
| 是否需要 y | 需要 | 不需要 |
| K 的含义 | 邻居数量 | 簇数量 |
| 距离对象 | 新样本与训练样本 | 样本与质心 |
| 结果 | 已知类别或数值 | 任意编号的簇 |
| 标准化 | 通常需要 | 通常需要 |

## 8. 常见错误清单

1. 把 KNN 的 `K` 和 K-means 的 `K` 混为一谈；
2. 未标准化量纲差异很大的特征；
3. 把类别编码后的数字直接当连续距离；
4. 只运行一次随机初始化；
5. 用簇编号直接和真实标签计算准确率；
6. 只追求更小的 inertia；
7. 忽略异常值对均值质心的拉动；
8. 强行用 K-means 处理月牙、环形或不同密度簇；
9. 对新数据重新拟合标准化器；
10. 得到簇后不检查质心、大小、稳定性和解释价值。

## 9. 适用条件

适合：

```text
数值特征
欧氏距离有意义
簇近似球形
簇尺度与密度较接近
需要快速、可扩展的中心型聚类
```

谨慎使用：

```text
大量类别特征
异常值很多
非凸形状
不同密度或大小差异很大
高维噪声严重
无法合理指定 K
```

## 10. 本章学习成果

完成本章后，应能够：

- 解释无监督学习、簇与质心；
- 手算距离、簇分配、质心更新和 SSE；
- 使用 NumPy 手写 K-means 主循环；
- 使用 sklearn 完成标准化、训练、预测和质心解释；
- 使用肘部法和轮廓系数辅助选择 K；
- 解释 `k-means++`、`n_init` 和局部最优；
- 判断异常值、非球形簇和高维特征带来的风险；
- 完成 Iris 无监督聚类并使用 ARI/NMI 事后评价。

## 11. 文件学习顺序

1. `KMeans学习路线.ipynb`
2. `KMeans第一课_聚类与基本思想.ipynb`
3. `KMeans第二课_距离与目标函数.ipynb`
4. `KMeans第三课_算法迭代与手写实现.ipynb`
5. `KMeans第四课_sklearn完整实战.ipynb`
6. `KMeans第五课_标准化与K值选择.ipynb`
7. `KMeans第六课_初始化与算法局限.ipynb`
8. `KMeans第七课_Iris综合案例.ipynb`
9. `KMeans总结笔记.ipynb`